# Week 4 L1 — Activity 2: Clean the assignment datasets

You now have two real datasets from the Office for National Statistics (ONS):

| File | What it holds |
|---|---|
| `life_expectancy_by_local_authority.csv` | life expectancy for every UK local authority, by sex and age group, for each three-year period from 2001-03 to 2017-19 |
| `gdhi_per_head_by_local_authority.csv` | gross disposable household income (GDHI) per person, in pounds, for every UK local authority, for each year from 1997 to 2024 |

You will clean them using the same steps you followed in Activity 1, `Week4_L1_Activity1_Clean_Two_Datasets.ipynb`. **Keep that notebook open beside this one.**

### How this notebook works

- The cells that load the files are already written. Run them first.
- After that, **you write the code**. Each step tells you which part of Activity 1 to look back at, gives you a hint if a tool is new, and shows what your result **might look like**.
- Your layout does not have to match exactly, but your **counts** (rows, columns, duplicates, missing values) should.
- Under every task there is an empty cell marked `# Your code here`. Add more cells whenever you need them.

By the end you will have saved two clean files:

| Clean file | What one row means |
|---|---|
| `life_expectancy_clean.csv` | one area × one period × one sex × one age group |
| `gdhi_per_head_clean.csv` | one area, with one column per year |

## What the numbers mean

Read this before you start: you cannot judge whether a value is wrong unless you know what it is supposed to measure.

**Life expectancy** here is the average number of *further* years a person in that age group could expect to live, if death rates stayed as they were in that period. So:

- for the age group `00-01` it is life expectancy at birth (around 75–85 years);
- for `90+` it is only a few years, because it counts the years *remaining*.

Small values are therefore not errors. Each value comes with a **95% confidence interval** (`Lower CI`, `Upper CI`): the range the true value very probably lies in. Smaller areas have wider intervals.

**GDHI per head** is the money households have left for spending or saving after taxes, National Insurance and pension contributions, plus benefits received, divided by the number of people living there. It is in pounds at the prices of each year (not adjusted for inflation).

---
# SetUp— Load the files

These cells are written for you. Keep both CSV files in the same folder as this notebook, then run each cell.

In [1]:
%pip install pandas numpy

from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path(".")
LIFE_FILE = DATA_DIR / "life_expectancy_by_local_authority.csv"
GDHI_FILE = DATA_DIR / "gdhi_per_head_by_local_authority.csv"

print(LIFE_FILE, "exists:", LIFE_FILE.exists())
print(GDHI_FILE, "exists:", GDHI_FILE.exists())


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
life_expectancy_by_local_authority.csv exists: True
gdhi_per_head_by_local_authority.csv exists: True


### Life expectancy: look, then read

In [2]:
pd.read_csv(LIFE_FILE, header=None, nrows=4)

,0,1,2,3,4,5,6,7,8,9,10
0,v4_2,Lower CI,Upper CI,two-year-intervals,Time,administrative-geography,Geography,sex,Sex,age-groups,AgeGroups
1,20.59,20.1,21.08,2002-04,2002-04,E07000129,Blaby,female,Female,65-69,65-69
2,38.76,38.23,39.29,2002-04,2002-04,E07000063,Lewes,female,Female,45-49,45-49
3,67.23,66.74,67.72,2002-04,2002-04,E07000011,Huntingdonshire,female,Female,15-19,15-19


In [3]:
life_raw = pd.read_csv(LIFE_FILE)
life_raw.head()

,v4_2,Lower CI,Upper CI,two-year-intervals,Time,administrative-geography,Geography,sex,Sex,age-groups,AgeGroups
0,20.59,20.10,21.08,2002-04,2002-04,E07000129,Blaby,female,Female,65-69,65-69
1,38.76,38.23,39.29,2002-04,2002-04,E07000063,Lewes,female,Female,45-49,45-49
2,67.23,66.74,67.72,2002-04,2002-04,E07000011,Huntingdonshire,female,Female,15-19,15-19
3,26.03,25.61,26.44,2002-04,2002-04,E07000045,Teignbridge,male,Male,55-59,55-59
4,68.05,67.72,68.39,2002-04,2002-04,E06000058,"Bournemouth, Christchurch and Poole",male,Male,10-14,10-14


### GDHI: look, then read

The top of this file has lines of different lengths, which `pd.read_csv` cannot preview. So we print the first lines as plain text instead.

In [4]:
with open(GDHI_FILE, encoding="utf-8") as f:
    for line_number in range(9):
        print(line_number, f.readline().rstrip()[:110])

0 
1 "Regional gross disposable household income (GDHI) at current basic prices"
2 "ONS Crown Copyright Reserved [from Nomis on 6 October 2026]"
3 "Analysis   :","standard"
4 "Component of GDHI:","Gross Disposable Household Income (GDHI)"
5 "Measure    :","GDHI per head (£)"
6 
7 "local authority: district / unitary (as of April 2023)","mnemonic","1997","1998","1999","2000","2001","2002",
8 


In [5]:
gdhi_raw = pd.read_csv(GDHI_FILE, skiprows=7)
gdhi_raw.head()

,local authority: district / unitary (as of April 2023),mnemonic,1997,1998,1999,2000,2001,2002,2003,2004,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Hartlepool,E06000001,8273.0,8315.0,8799.0,9110.0,9558.0,9914.0,10113.0,10492.0,...,14749.0,14996.0,15236.0,15828.0,16233.0,16455.0,17292.0,17957.0,19095.0,20080.0
1,Middlesbrough,E06000002,8645.0,8325.0,8818.0,9265.0,9704.0,9793.0,10020.0,10535.0,...,14498.0,14498.0,14663.0,15132.0,15711.0,15816.0,16202.0,16645.0,17134.0,17855.0
2,Redcar and Cleveland,E06000003,8876.0,8567.0,9092.0,9579.0,10057.0,10126.0,10390.0,10749.0,...,15183.0,15322.0,15591.0,16032.0,16484.0,16673.0,16873.0,17644.0,18672.0,19739.0
3,Stockton-on-Tees,E06000004,9474.0,9358.0,9805.0,10201.0,10676.0,10975.0,11240.0,11880.0,...,15571.0,15768.0,16070.0,16244.0,17174.0,17457.0,17896.0,18477.0,19581.0,20623.0
4,Darlington,E06000005,9407.0,9319.0,9513.0,10250.0,11160.0,11118.0,11394.0,11795.0,...,15381.0,15915.0,16593.0,17038.0,17679.0,18097.0,18415.0,19068.0,20457.0,21587.0


> **Question.** Using the printout above, explain why the read uses `skiprows=7`. Which line number holds the column names, and what is on the lines above it?

*Your answer:*

## Bring your two functions across

Copy `quick_audit` from **"Wrapping repeated work in a function"** and `clean_headers` from **Step 2** of Activity 1 into the cell below, then run it.

Remember: running a cell that only *defines* functions prints nothing.

In [7]:
def quick_audit(df,name):
    print(name)
    print("Rows and columns:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Type and missing values per column:")
    summary = pd.DataFrame({"type": df.dtypes, "missing": df.isna().sum()})
    print(summary)

---
# Part 1 — Life expectancy

## Step 1: First look

Call `quick_audit` on `life_raw`.

*Activity 1: "First look: four questions to ask of any table".*

In [8]:
quick_audit(life_raw, "Raw life data")

Raw life data
Rows and columns: (286280, 11)
Duplicate rows: 0
Type and missing values per column:
                             type  missing
v4_2                      float64        0
Lower CI                  float64        0
Upper CI                  float64        0
two-year-intervals            str        0
Time                          str        0
administrative-geography      str        0
Geography                     str        0
sex                           str        0
Sex                           str        0
age-groups                    str        0
AgeGroups                     str        0


Your result might look something like this:

```
Raw life expectancy data
Rows and columns: (286280, 11)
Duplicate rows: 0
Type and missing values per column:
                             type  missing
v4_2                      float64        0
Lower CI                  float64        0
Upper CI                  float64        0
two-year-intervals         object        0
Time                       object        0
administrative-geography   object        0
Geography                  object        0
sex                        object        0
Sex                        object        0
age-groups                 object        0
AgeGroups                  object        0
```

(Newer versions of pandas show `str` instead of `object` for text columns.)

No missing values and no duplicates. That does **not** mean the file is clean. Look at the column names again.

> **Question.** Which columns look as if they hold the same information twice?

*Your answer:*

In [9]:
print(life_raw.duplicated().sum())

0


## Step 2: Find the columns that repeat each other

There are three suspicious pairs:

| Pair | |
|---|---|
| `two-year-intervals` and `Time` | |
| `sex` and `Sex` | |
| `age-groups` and `AgeGroups` | |

For each pair, use `.value_counts()` on both columns and compare them (*Activity 1, Step 4b*).

**New tool.** To count the rows where two columns disagree, compare them with `!=` and add up the `True`s, just as you counted negative waits in *Step 5c*:

```python
(life_raw["column_a"] != life_raw["column_b"]).sum()
```

print the top 10 values of the life_raw data frame to find the difference between eachpair

In [14]:
[life_raw["two-year-intervals"] != life_raw["Time"]]

[0         False
 1         False
 2         False
 3         False
 4         False
           ...  
 286275    False
 286276    False
 286277    False
 286278    False
 286279    False
 Length: 286280, dtype: bool]

Your findings might look like this:

```
two-year-intervals vs Time:      0 rows differ      → identical
sex vs Sex:                 286280 rows differ      → only the capital letter differs
age-groups vs AgeGroups:     42942 rows differ      → '0-1' vs '00-01', '1-4' vs '01-04', '5-9' vs '05-09'
```

In every pair, both columns mean the same thing. Keep **one** of each and drop the other.

> **Question.** `age-groups` writes `5-9`; `AgeGroups` writes `05-09`. Sort the values of each column in your head, alphabetically. Where does `5-9` end up, compared with `50-54`? Which column should you keep, and why?

*Your answer:*

## Step 3: Give the columns clear names

First, **try** `clean_headers` on a copy, just to see what it produces (*Activity 1, Step 2*).

In [27]:
def clean_header(columns):
    names = columns.str.strip()
    names =names.str.lower()
    names = names.str.replace(r"[^a-zA-Z0-9_]+", "_", regex=True)
    names = names.str.strip("_")
    return names

In [28]:
life = life_raw.copy()
life.columns = clean_header(life.columns)
life.columns

Index(['v4_2', 'lower_ci', 'upper_ci', 'two_year_intervals', 'time',
       'administrative_geography', 'geography', 'sex', 'sex', 'age_groups',
       'agegroups'],
      dtype='str')

You should notice two problems:

```
Index(['v4_2', 'lower_ci', 'upper_ci', 'two_year_intervals', 'time',
       'administrative_geography', 'geography', 'sex', 'sex', 'age_groups',
       'agegroups'], dtype='object')
```

- `v4_2` tells a reader nothing about what the column holds.
- There are now **two columns called `sex`**. pandas allows it, but `df["sex"]` would return both, and most code that follows would break.

So for this file, tidying the format of the names is not enough: they need **new** names.

Make a working copy called `life` (remember `.copy()`), then:

1. **Drop** the three columns you decided not to keep.
2. **Rename** the rest using a dictionary, `{old name: new name}`, exactly like the translation dictionaries in *Step 4d*.

**New tools:**

```python
table = table.drop(columns=["column_to_remove", "another_one"])
table = table.rename(columns={"old_name": "new_name", "old_2": "new_2"})
```

Suggested new names:

| Old name | New name |
|---|---|
| `v4_2` | `life_expectancy` |
| `Lower CI` | `lower_ci` |
| `Upper CI` | `upper_ci` |
| `Time` | `period` |
| `administrative-geography` | `area_code` |
| `Geography` | `area_name` |
| `Sex` | `sex` |
| `AgeGroups` | `age_group` |

In [ ]:
life # this is your current dataframe

# Remove the duplicated 'sex' column created when the headers were cleaned
life = life.loc[:, ~life.columns.duplicated()].copy()

# Drop columns that are repeated versions of other fields
life = life.drop(columns=["two_year_intervals", "agegroups"])

life = life.rename(columns={
    "v4_2": "life_expectancy",
    "time": "period",
    "administrative_geography": "area_code",
    "geography": "area_name",
    "age_groups": "age_group"
})

In [30]:
life.head()

,life_expectancy,lower_ci,upper_ci,two_year_intervals,time,area_code,area_name,sex,sex,age_groups,age_group
0,20.59,20.10,21.08,2002-04,2002-04,E07000129,Blaby,female,Female,65-69,65-69
1,38.76,38.23,39.29,2002-04,2002-04,E07000063,Lewes,female,Female,45-49,45-49
2,67.23,66.74,67.72,2002-04,2002-04,E07000011,Huntingdonshire,female,Female,15-19,15-19
3,26.03,25.61,26.44,2002-04,2002-04,E07000045,Teignbridge,male,Male,55-59,55-59
4,68.05,67.72,68.39,2002-04,2002-04,E06000058,"Bournemouth, Christchurch and Poole",male,Male,10-14,10-14


Your columns might look like this:

```
Index(['life_expectancy', 'lower_ci', 'upper_ci', 'period', 'area_code',
       'area_name', 'sex', 'age_group'], dtype='object')
```

You can reorder them so the identifying columns come first, by selecting them in the order you want:

```python
life = life[["area_code", "area_name", "period", "sex", "age_group",
             "life_expectancy", "lower_ci", "upper_ci"]]
```

In [ ]:
life = life[["area_code", "area_name", "time", "sex", "age_group","life_expectancy", "lower_ci", "upper_ci"]]

KeyError: "['two_year_intervals', 'agegroups'] not found in axis"

In [45]:
life.head()

,area_code,area_name,time,sex,age_group,life_expectancy,lower_ci,upper_ci
0,E07000129,Blaby,2002-04,female,65-69,20.59,20.10,21.08
1,E07000063,Lewes,2002-04,female,45-49,38.76,38.23,39.29
2,E07000011,Huntingdonshire,2002-04,female,15-19,67.23,66.74,67.72
3,E07000045,Teignbridge,2002-04,male,55-59,26.03,25.61,26.44
4,E06000058,"Bournemouth, Christchurch and Poole",2002-04,male,10-14,68.05,67.72,68.39


## Step 4: Decide what one row means (the grain)

*Activity 1, Step 3.*

> **Question.** Which four columns together should identify exactly one row? Write the grain as a sentence: "one row is one ___ × one ___ × one ___ × one ___".

*Your answer:*

Store those four column names in a list called `grain`, then count how many rows share the same key with `.duplicated(grain).sum()` (*Activity 1, Step 9*).

In [47]:
life.to_csv("life_expectancy_by_local_authority_cleaned.csv", index=False)
print("Saved data/life_expectancy_by_local_authority_cleaned.csv",life.shape)

Saved data/life_expectancy_by_local_authority_cleaned.csv (286280, 8)


Expected:

```
Rows sharing the same key: 0
```

## Step 5: Check the text columns

*Activity 1, Steps 4a and 4b.*

1. Strip spaces from the five text columns (`area_code`, `area_name`, `period`, `sex`, `age_group`) with the same `for` loop.
2. Use `.value_counts(dropna=False)` on `sex`, `age_group` and `period`.

This file turns out to be tidy here. Checking still matters: "I checked and found nothing" is a result you can report; "I didn't check" is not.

In [49]:
text_columns = ["area_code", "area_name","time", "sex", "age_group"]

for column in text_columns:
    life[column] = life[column].astype("string").str.strip()

life[text_columns]

,area_code,area_name,time,sex,age_group
0,E07000129,Blaby,2002-04,female,65-69
1,E07000063,Lewes,2002-04,female,45-49
2,E07000011,Huntingdonshire,2002-04,female,15-19
3,E07000045,Teignbridge,2002-04,male,55-59
4,E06000058,"Bournemouth, Christchurch and Poole",2002-04,male,10-14
...,...,...,...,...,...
286275,E08000010,Wigan,2017-19,male,30-34
286276,S12000034,Aberdeenshire,2017-19,male,35-39
286277,W06000004,Denbighshire,2017-19,male,85-89
286278,S12000014,Falkirk,2017-19,male,50-54


In [51]:
life["age_group"].value_counts(dropna=False)

age_group
65-69    14314
45-49    14314
15-19    14314
55-59    14314
10-14    14314
75-79    14314
35-39    14314
20-24    14314
50-54    14314
70-74    14314
25-29    14314
85-89    14314
01-04    14314
40-44    14314
80-84    14314
05-09    14314
90+      14314
00-01    14314
30-34    14314
60-64    14314
Name: count, dtype: Int64

In [52]:
life["time"].value_counts(dropna=False)

time
2002-04    16840
2001-03    16840
2003-05    16840
2004-06    16840
2005-07    16840
2006-08    16840
2007-09    16840
2008-10    16840
2009-11    16840
2010-12    16840
2011-13    16840
2012-14    16840
2013-15    16840
2014-16    16840
2015-17    16840
2016-18    16840
2017-19    16840
Name: count, dtype: Int64

In [50]:
life["sex"].value_counts(dropna=False)

sex
female    143140
male      143140
Name: count, dtype: Int64

What you might see:

```
sex
Female    143140
Male      143140

age_group                          period
00-01    14314                     2001-03    16840
01-04    14314                     2002-04    16840
05-09    14314                     ...
...                                2017-19    16840
90+      14314                     (17 periods)
(20 age groups)
```

Every group has the same count. That is a good sign: every area has a value for every combination.

## Step 6: Keep only one kind of area

This is the most important step for this file.

Look at how many different area codes there are, and at the first three characters of each code.

**New tool.** `.str[:3]` takes the first three characters of every value:

```python
life["area_code"].str[:3].value_counts().sort_index()
```

In [53]:
life["area_code"].str[:3].value_counts().sort_index()

area_code
E06     38080
E07    127840
E08     24480
E09     21760
E10     17000
E11      4080
E12      6120
E92       680
N09      7480
N92       680
S12     21760
S92       680
W06     14960
W92       680
Name: count, dtype: Int64

You should see:

```
E06     38080        E12      6120
E07    127840        E92       680
E08     24480        N09      7480
E09     21760        N92       680
E10     17000        S12     21760
E11      4080        S92       680
                     W06     14960
                     W92       680
```

The first three characters of an ONS code say **what kind of area** it is:
https://www.ons.gov.uk/methodology/geography/ukgeographies/administrativegeography/england
| Prefix | Kind of area | Keep? |
|---|---|---|
| `E06` | English unitary authority | local authority |
| `E07` | English district | local authority |
| `E08` | English metropolitan district | local authority |
| `E09` | London borough | local authority |
| `W06` | Welsh unitary authority | local authority |
| `S12` | Scottish council area | local authority |
| `N09` | Northern Ireland district | local authority |
| `E10` | English county | larger area made of districts |
| `E11` | English metropolitan county | larger area made of districts |
| `E12` | English region | larger area |
| `E92`, `W92`, `S92`, `N92` | a whole country | larger area |

So the file mixes **local authorities** with the **larger areas they add up to**. England appears once as `E92`, and again as every district inside it.

> **Question.** Why would keeping both levels be a problem if you later averaged life expectancy across all rows?

*Your answer:*

> **Question.** Before you remove anything, find the rows where `area_name` is `"West Midlands"`. How many different `area_code`s does that name have? What does this tell you about using **names** rather than **codes** to identify an area?

*Your answer:*

In [ ]:
# Your code here

Now keep **only** the local authority rows.

**New tool.** `.isin(list)` gives `True` for each value that appears in the list. Use it as a condition to keep rows, the same way you used a condition in *Step 6* of Activity 1 to show duplicates:

```python
local_authority_prefixes = ["E06", "E07", "E08", "E09", "W06", "S12", "N09"]
is_local_authority = life["area_code"].str[:3].isin(local_authority_prefixes)
life = life[is_local_authority]
```

Print the number of rows before and after, as in *Step 6* of Activity 1, and count the distinct areas with `.nunique()`.

In [ ]:
rows_before = len(life)
life

In [55]:
life["area_code"].value_counts()

area_code
E07000129    680
E07000063    680
E07000011    680
E07000045    680
E06000058    680
            ... 
E07000237    680
E07000196    680
E12000004    680
N09000010    680
E10000003    680
Name: count, Length: 421, dtype: Int64

In [54]:
life["area_code"].nunique()

421

Expected:

```
Rows before: 286280
Rows after:  256360
Removed:     29920
Distinct local authorities: 377
```

Check: 377 areas × 17 periods × 2 sexes × 20 age groups = 256,360. Every area is complete.

## Step 7: Check the numbers make sense

*Activity 1, Steps 5 and 5c.*

The three number columns are already `float64`, so no conversion is needed. That does not mean every value is believable. Check:

1. the smallest and largest `life_expectancy` (`.min()`, `.max()`);
2. how many rows break the rule **lower_ci ≤ life_expectancy ≤ upper_ci**. Build two conditions, one for each side, and count the rows where either is `True` (`|` means "or").

In [ ]:
# Your code here

Expected:

```
Smallest life expectancy: 2.43
Largest life expectancy:  87.22
Rows outside their confidence interval: 0
```

> **Question.** 2.43 years looks alarming. Which age group do you think it belongs to? Check, and explain why it is not an error (re-read "What the numbers mean").

*Your answer:*

In [ ]:
# Your code here

## Step 8: Missing values and duplicates

*Activity 1, Steps 6 and 7.*

Count missing values per column and exact duplicate rows on your cleaned table.

In [ ]:
# Your code here

Expected: zero missing in every column, zero duplicate rows. There is nothing to fill, so the decision to record is simply that no values needed filling.

## Step 9: Sort, check and save

*Activity 1, Steps 8, 9 and 10.*

1. Sort by your `grain` and reset the index.
2. Write `assert` checks for at least: no repeated keys; no missing area codes; every area code starts with a local authority prefix; every value lies inside its interval.
3. Run `quick_audit` again and compare with Step 1.
4. Save as **`life_expectancy_clean.csv`**.

In [ ]:
# Your code here

Your clean table might look like this:

```
   area_code   area_name   period     sex age_group  life_expectancy  lower_ci  upper_ci
0  E06000001  Hartlepool  2001-03  Female     00-01            78.40     77.71     79.08
1  E06000001  Hartlepool  2001-03  Female     01-04            77.81     77.18     78.43
2  E06000001  Hartlepool  2001-03  Female     05-09            73.81     73.18     74.43
3  E06000001  Hartlepool  2001-03  Female     10-14            68.81     68.18     69.43
4  E06000001  Hartlepool  2001-03  Female     15-19            63.81     63.18     64.43
...

Saved life_expectancy_clean.csv: (256360, 8)
```

Notice that `00-01`, `01-04`, `05-09` come out in the right order. Compare with your answer to the question in Step 2.

---
# Part 2 — GDHI per head

## Step 1: First look

Call `quick_audit` on `gdhi_raw`. Then look at the **last** five rows with `.tail()`: problems often hide at the bottom of an export.

In [ ]:
# Your code here

Your result might look something like this:

```
Rows and columns: (364, 30)
Duplicate rows: 0
                                                         type  missing
local authority: district / unitary (as of April 2023)  object        2
mnemonic                                                 object        1
1997                                                    float64        2
1998                                                    float64        2
...
2024                                                    float64        2
```

and the tail:

```
    local authority: district / ...    mnemonic                                     1997     1998
360  Ards and North Down               N09000011                                     9397.0   9409.0
361  Column Total                      Column Total                                 10610.0  10827.0
362  NaN                               Column totals may not sum due to rounding.      NaN      NaN
363  NaN                               NaN                                             NaN      NaN
```

> **Question.** The bottom three rows are not local authorities. What is each one?

*Your answer:*

## Step 2: Give the columns clear names

The first column has a long, awkward name and `mnemonic` is not a word most readers know. Make a working copy called `gdhi`, then rename:

| Old name | New name |
|---|---|
| the first column | `area_name` |
| `mnemonic` | `area_code` |

**Hint.** The first column's name is long; you don't have to type it. `gdhi.columns[0]` gives it to you, so you can use that as the key in the dictionary.

Leave the year columns (`"1997"` … `"2024"`) as they are; they are already short and clear.

*Activity 1, Step 2, and Step 3 of Part 1 above.*

In [ ]:
# Your code here

Your first columns might look like this:

```
Index(['area_name', 'area_code', '1997', '1998', '1999', ...
```

## Step 3: Remove the rows that are not areas

Use the **same prefix check** as Step 6 of Part 1. It keeps the 361 local authorities and removes the total, the footnote and the blank row in one go. Print rows before and after.

In [ ]:
# Your code here

Expected:

```
Rows before: 364
Rows after:  361
Removed:     3
```

> **Question.** Why is it better to keep rows that match a rule ("the code starts with a local authority prefix") than to delete the three bad rows by their row numbers?

*Your answer:*

## Step 4: Decide what one row means

This file is shaped differently from the life expectancy file. Each **row** is one area, and each **year** has its own column.

> **Question.** Write the grain of this table. Then say what a single **cell** in the `2019` column represents.

*Your answer:*

Check that no `area_code` appears twice (`.duplicated()` on that one column, then `.sum()`).

In [ ]:
# Your code here

Expected:

```
Repeated area codes: 0
```

## Step 5: Check the text columns

Strip spaces from `area_name` and `area_code`, as in *Activity 1, Step 4a*.

In [ ]:
# Your code here

## Step 6: Turn the year columns into whole numbers

GDHI per head is a whole number of pounds, yet the year columns were read as `float64`.

> **Question.** Why did pandas store them as decimals? (Look back at what you removed in Step 3.)

*Your answer:*

Now that those rows are gone, convert every year column to `Int64` (*Activity 1, Step 5a*). There are 28 of them, so use a `for` loop (*Step 4a*).

**Hint.** `gdhi.columns[2:]` gives every column name from the third onwards: exactly the year columns.

Then check the smallest value across all years, to make sure nothing is zero or negative.

In [ ]:
# Your code here

Expected:

```
area_name    string
area_code    string
1997          Int64
1998          Int64
...
2024          Int64

Smallest GDHI per head in any year: 5738
```

## Step 7: Missing values

Count missing values across the whole table.

In [ ]:
# Your code here

Expected: zero. Every local authority has a value for every year.

## Step 8: Sort, check and save

1. Sort by `area_code` and reset the index.
2. Write `assert` checks: no repeated area codes; every code starts with a local authority prefix; no missing values; every value greater than zero.
3. Save as **`gdhi_per_head_clean.csv`**.

In [ ]:
# Your code here

Your clean table might look like this:

```
              area_name  area_code  1997  1998  1999  2000  ...  2024
0            Hartlepool  E06000001  8273  8315  8799  9110  ...  20080
1         Middlesbrough  E06000002  8645  8325  8818  9265  ...  17855
2  Redcar and Cleveland  E06000003  8876  8567  9092  9579  ...  19739
3      Stockton-on-Tees  E06000004  9474  9358  9805 10201  ...  20623
...

Saved gdhi_per_head_clean.csv: (361, 30)
```

---
## Cleaning log

Fill in one row for every change you made, and one for every check that found nothing. A reader should be able to repeat your cleaning from this table alone.

| File | Step | What you found | What you did | Rows before | Rows after |
|---|---|---|---|---|---|
| life expectancy | 2 | three pairs of columns hold the same information | | | |
| life expectancy | 6 | | | | |
| | | | | | |
| GDHI | 3 | | | | |
| | | | | | |

## Exit ticket

1. The life expectancy file had no missing values and no duplicates, yet still needed cleaning. Name the two biggest problems you fixed.
2. Why are area **codes** a safer identifier than area **names**? Give the example you found.
3. The two clean files have different grains. Write both, and describe in one sentence how their shapes differ.
4. Life expectancy covers 2001-03 to 2017-19; GDHI covers 1997 to 2024. Life expectancy has 377 local authorities; GDHI has 361. Note down one possible reason for each difference.